In [31]:
import pandas as pd
from splinter import Browser
from bs4 import BeautifulSoup as bs
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import sys
import requests

In [32]:
url='https://booknow.appointment-plus.com/b6l56ers/'
executable_path = {'executable_path':'chromedriver.exe'}
browser = Browser('chrome',  headless=False)

In [33]:
browser.visit(url)

input_element = browser.find_by_name('loginname')   
input_element.fill('jmedias')

input_element = browser.find_by_name('password')   
input_element.fill('3ggR0ll$')

button_element = browser.find_by_value('Log In').click()

In [41]:
browser.find_by_name('e_id').click()
browser.find_by_value('5311').click()

browser.find_by_name('service_id').click()
browser.find_by_value('6453').click()

if browser.is_element_not_present_by_id('cv-leftnav-item-calendar-available-id'):
    browser.find_by_name('e_id').click()
    browser.find_by_value('4807').click()
    if browser.is_element_not_present_by_id('cv-leftnav-item-calendar-available-id'):
        browser.quit
        sys.exit('exit')
    
        
    if browser.find_by_id('cv-leftnav-item-calendar-available-id'):
        browser.find_by_id('cv-leftnav-item-calendar-available-id').click()   
    


SystemExit: exit

In [ ]:
#if browser.find_by_id('cv-leftnav-item-calendar-available-id'):


In [ ]:
html = browser.html
soup = bs(html, "html.parser")

In [ ]:
date_scheduled = soup.find_all('table', class_='appointment-list-style')

date_list =[]
for TDs in date_scheduled:
        innerDivs= TDs.find_all("td")
        for idiv in innerDivs:
                date_list.append(idiv.get_text(strip=True))

In [ ]:
if browser.is_element_present_by_name('next'):
        browser.find_by_name('next').click()
        
        html = browser.html
        soup = bs(html, "html.parser")
        date_scheduled = soup.find_all('table', class_='appointment-list-style')
        
        for TDs in date_scheduled:
                innerDivs= TDs.find_all("td")
                for idiv in innerDivs:
                        date_list.append(idiv.get_text(strip=True))

In [ ]:
df = pd.DataFrame(date_list)

day_date=df[df[0].str.contains(',', na=False)]
day_date = day_date.reset_index(drop=True)
day_date.rename(columns = {0:'Date'}, inplace=True)

KeyError: 0

In [ ]:
time_slot=df[df[0].str.contains(':', na=False)]
time_slot = time_slot.reset_index(drop=True)
time_slot.rename(columns= {0:'Time'}, inplace=True)

In [ ]:
court2df = day_date.join(time_slot)
court2df['Court Name'] = 'Court 2'
court2df['Court ID'] = '5311' 
court2df = court2df[['Court Name', 'Date', 'Time','Court ID']]

In [ ]:
browser.find_by_name('e_id').click()
browser.find_by_value('4807').click()

In [ ]:
html = browser.html
soup = bs(html, "html.parser")

In [ ]:
# find the table tag with the class 'appointment-list-style'
# then create an empty list called date_list
# get all the text that is in between the <td></td> tags
# and enter it ino the date_list list. 

date_scheduled = soup.find_all('table', class_='appointment-list-style')

date_list =[]
for TDs in date_scheduled:
        innerDivs= TDs.find_all("td")
        for idiv in innerDivs:
                date_list.append(idiv.get_text(strip=True))

In [ ]:
# if the next button is available, 
# click it and grab all the text that is in between <td>,/td> 
# and add it to the date_list list. 

if browser.is_element_present_by_name('next'):
        browser.find_by_name('next').click()
        
        html = browser.html
        soup = bs(html, "html.parser")
        date_scheduled = soup.find_all('table', class_='appointment-list-style')
        
        for TDs in date_scheduled:
                innerDivs= TDs.find_all("td")
                for idiv in innerDivs:
                        date_list.append(idiv.get_text(strip=True))

In [ ]:
df = pd.DataFrame(date_list)

# extract all the rows that contain a ',' from the df and save to the day_date df.Date
# reset the index and rename the column from 0 to Date. 

day_date = df[df[0].str.contains(',', na=False)]
day_date = day_date.reset_index(drop=True)
day_date.rename(columns = {0:'Date'}, inplace=True)

In [ ]:
# extract all the rows that contain ':' and save it to the time_slot df.  
# reset the index and rename the column from 0 to Time. 

time_slot = df[df[0].str.contains(':', na=False)]
time_slot = time_slot.reset_index(drop=True)
time_slot.rename(columns= {0:'Time'}, inplace=True)

In [ ]:
# create court2df by joining the day_date and time_slot together as two separate columns
# create a new column called Court Name. This will be used for the text message.
# create a new column called Court ID. This will be used by splinter when booking in the 
#  other program. 

court1df = day_date.join(time_slot)
court1df['Court Name'] = 'Court 2' 
court1df['Court ID'] = '4807'
court1df = court1df[['Court Name', 'Date', 'Time','Court ID']]


In [ ]:
# add the court2df to the bottom of court1df. 
# then create a new column in the datetime object format using the Time column
# then drop the date from the time_obj column
# sort the rows based on time_obj

courtsdf = pd.concat([court1df,court2df])
courtsdf = courtsdf.reset_index(drop=True)
courtsdf['time_obj'] = pd.to_datetime(courtsdf['Time'])
courtsdf['time_obj'] = pd.to_datetime(courtsdf['time_obj'],format='%H:%M').dt.time
courtsdf.sort_values(by='time_obj', inplace=True)


In [ ]:
# Select only the rows containing the word, Sunday or Thursday from the dataframe
# and save it to filtered_dates.

filtered_dates = courtsdf[courtsdf['Date'].str.contains('Thursday|Sunday', case=False, regex=True)]
filtered_dates = filtered_dates.reset_index(drop=True)
filtered_dates = filtered_dates.drop('Date', axis=1)
filtered_dates

,Court Name,Time,Court ID,time_obj
0,Court 1,12:00pm,4807,12:00:00
1,Court 2,12:00pm,5311,12:00:00
2,Court 1,12:30pm,4807,12:30:00
3,Court 2,12:30pm,5311,12:30:00
4,Court 1,1:00pm,4807,13:00:00


In [ ]:
# test the filtered dates DF to see if it contains the time slots we are sniping. 
# if it exists, extract those rows and save it to SMSdf.  
# then send a text to the numbers for availability notification.

if filtered_dates['Time'].str.contains('7:00pm|1:30pm', case=False, regex=True).any():

    SMSdf = filtered_dates[filtered_dates['Time'].str.contains('7:00pm|1:30pm', case=False, regex=True)]
    SMSdf = SMSdf.reset_index(drop=True).drop('time_obj',axis=1).drop('Court ID', axis=1)
    
    
    print("The following times are now available:")
    print(SMSdf)

The following times are now available:
  Court Name     Time
0    Court 1  12:00pm
1    Court 2  12:00pm
2    Court 1  12:30pm
3    Court 2  12:30pm


In [ ]:
    email = "picklebotSL@gmail.com"
    pas = "cupN00dles"

In [ ]:
    sms_gateway = '8327154545@txt.att.net'

    smtp = "smtp.gmail.com"
    port = 587

In [ ]:
    server = smtplib.SMTP(smtp,port)


    server.starttls()

(220, b'2.0.0 Ready to start TLS')

In [ ]:
    server.login(email,pas)

In [ ]:
    msg = MIMEMultipart()
    msg['From'] = email
    msg['To'] = sms_gateway

In [ ]:
    msg['Subject'] = "Available Time Slots\n"

In [ ]:
    body = SMSdf

In [ ]:
    msg.attach(MIMEText(body, 'plain'))

    sms = msg.as_string()

    server.sendmail(email,sms_gateway,sms)

    # lastly quit the server
    server.quit()